# Baseline / contextual decomposition of the final-readout margin

Organizing principle: final selection at the first answer token depends on a context-independent
readout **baseline** plus an item-specific **contextual** update. The margin decomposes EXACTLY:

$$M_i(a_i,b_i) = \underbrace{(b_{a_i}-b_{b_i})}_{\text{baseline margin}} + \underbrace{(c_{i,a_i}-c_{i,b_i})}_{\text{contextual margin}}$$

where $b_v = W_U[v]\cdot\bar h$ (baseline) and $c_i(v)=W_U[v]\cdot(h_i-\bar h)$ (contextual), with
$\bar h$ a **leave-one-out** mean of the final residual (so item $i$ never contributes to its own
baseline).

Two regimes (no competition metaphor):
- **answer-support limited**: the contextual answer signal is the shortfall; answer-up is enough.
- **baseline-limited**: the selected alternative carries too much readout baseline, so answer-up alone
  is insufficient.

Alignment tests (how the baseline becomes operational): cos(mean residual, frequency direction),
corr(baseline $b_v$, token frequency), corr($b_v$, $W_U[v]\cdot r$), cos(mean residual, BOS/sink dir).

**Toned-down thesis:** *factual errors at the first answer token often occur when the contextual
answer signal fails to overcome a frequency-graded final-readout baseline.* The baseline term may be
**one way** the frequency direction becomes operational at readout (not necessarily the whole story).

Needs `rw_core.py`, `mech_core.py`, `mech_runner.py`, `baseline_core.py`. Run `test_baseline_core.py`.

## 0. Config + data

In [ ]:
import numpy as np, json, gc, torch, re
import rw_core as rw, mech_core as mc, mech_runner as mr, baseline_core as bc
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
QA="Answer with a short factual answer.\nQuestion: {q}\nAnswer:"
TEMPLATES=[QA,"Q: {q}\nA:","Please answer concisely.\n{q}\nAnswer:","{q} The answer is"]
N_ITEMS=400; CAP=300

MODELS=[
 "meta-llama/Llama-3.1-8B","meta-llama/Llama-3.2-3B","meta-llama/Llama-3.2-1B",
 "meta-llama/Llama-3.2-3B-Instruct","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-3B-Instruct",
 "Qwen/Qwen2.5-7B","mistralai/Mistral-7B-v0.1",
]

ds=load_dataset("akariasai/PopQA",split="test")
def aliases(r):
    a=r["possible_answers"]
    if isinstance(a,str):
        try: a=json.loads(a)
        except: a=[a]
    return a
ITEMS=[{"q":str(r["question"]),"gold":aliases(r),"rel":str(r.get("prop","na"))} for r in ds]
REL=defaultdict(list)
for it in ITEMS:
    for a in it["gold"]: REL[it["rel"]].append(a)
np.random.default_rng(0).shuffle(ITEMS); ITEMS=ITEMS[:N_ITEMS]
print("items per model:",len(ITEMS))

## 1. Run the decomposition on all models

In [ ]:
BASE={}
for name in MODELS:
    print("="*70); print(name,flush=True)
    try:
        ctx=mr.make_ctx(name,DEVICE,DTYPE,ITEMS,REL,QA,TEMPLATES)
        BASE[name]=mr.exp_baseline_decomposition(ctx, ctx["freq"], max_items=CAP)
        r=BASE[name]
        print(f"  n={r.get('n')} base_margin={r.get('mean_baseline_margin',float('nan')):+.3f} "
              f"ctx_margin={r.get('mean_contextual_margin',float('nan')):+.3f} "
              f"baseline_limited={r.get('frac_baseline_limited',float('nan')):.2f} "
              f"exact={r.get('exactness',float('nan')):.1e}",flush=True)
    except Exception as e:
        import traceback; traceback.print_exc(); BASE[name]={"status":"error","error":f"{type(e).__name__}: {e}"}
    finally:
        try: mr.free_ctx(ctx); del ctx
        except Exception: pass
        torch.cuda.empty_cache(); gc.collect()
json.dump(BASE,open("baseline_decomposition.json","w"),indent=2,default=float)
print("\nsucceeded:",len([k for k,v in BASE.items() if "status" not in v]),"/",len(BASE))

## 2. The key table — baseline vs contextual margin, regimes, recovery
`exactness` should be ~1e-6 or smaller (the decomposition is algebraically exact). A more negative
`base_margin` means the selected alternative carries more readout baseline than the answer.

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["n"],
        "base_margin":round(v["mean_baseline_margin"],3),
        "ctx_margin":round(v["mean_contextual_margin"],3),
        "final_margin":round(v["mean_final_margin"],3),
        "frac_base_neg":round(v["frac_baseline_negative"],3),
        "frac_baseline_limited":round(v["frac_baseline_limited"],3),
        "answer_up":round(v["answer_up_recovery"],3),
        "both":round(v["both_recovery"],3),
        "exact":f"{v['exactness']:.1e}"}
       for nm,v in BASE.items() if "status" not in v]
df=pd.DataFrame(rows); print(df.to_string(index=False))
print("\nif Qwen2.5-3B has a more negative base_margin and higher frac_baseline_limited than")
print("Llama/Mistral, that explains why answer-up alone fails there (baseline-limited regime).")

## 3. Highlight: is Qwen2.5-3B baseline-limited where Llama/Mistral are answer-support-limited?

In [ ]:
sub=[r for r in rows]
if sub:
    df2=pd.DataFrame(sub)[["model","base_margin","frac_baseline_limited","answer_up","both"]]
    df2=df2.sort_values("base_margin")
    print(df2.to_string(index=False))
    print("\nread bottom (most negative base_margin) vs top: the baseline-limited models should be the")
    print("ones where answer_up << both (joint intervention needed).")

## 4. Alignment — how does the baseline become operational?
cos(mean residual, frequency direction), corr(baseline, token frequency), corr(baseline, unembedding
frequency projection), and cos(mean residual, BOS/sink direction). These test whether the baseline is
**one way** the frequency direction acts at readout (not a claim that it is the whole story).

In [ ]:
rows=[{"model":nm.split("/")[-1],
        "cos(meanres,freqdir)":round(v.get("cos_meanres_freqdir",float("nan")),3),
        "corr(base,freq)":round(v.get("corr_baseline_freq",float("nan")),3),
        "corr(base,unembed_freqproj)":round(v.get("corr_baseline_unembedfreqproj",float("nan")),3),
        "cos(meanres,bosdir)":round(v.get("cos_meanres_bosdir",float("nan")),3)}
       for nm,v in BASE.items() if "status" not in v]
print(pd.DataFrame(rows).to_string(index=False))
print("\nstrong cos(meanres,freqdir) and corr(base,freq) support: the baseline is frequency-graded and")
print("is one route by which the frequency direction becomes operational at readout. cos(meanres,bos)")
print("links the baseline to the sink/BOS read.")

## Notes (framing)
- The decomposition is exact (`exactness` ~ machine precision); the *interpretation* of $\bar h$ as a
  context-independent default depends on the mean estimate, which is leave-one-out here so no item
  contributes to its own baseline.
- Regimes: **answer-support limited** (answer-up suffices) vs **baseline-limited** (selected alternative
  has too much readout baseline; answer-up alone insufficient). Avoid 'competition-limited.'
- The baseline alignment (section 4) is what licenses connecting the baseline to frequency. Until that
  is strong, say: *the baseline term may be one way the frequency direction becomes operational at
  readout.*
- Thesis (toned down): *factual errors at the first answer token often occur when the contextual answer
  signal fails to overcome a frequency-graded final-readout baseline.*
- Raise `N_ITEMS`/`CAP` for final numbers.